In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import gc
import torch

# Delete any old model variables if they exist in the namespace
if 'model' in locals():
    del model
if 'optimizer' in locals():
    del optimizer

# Force garbage collection and empty the PyTorch cache
gc.collect()
torch.cuda.empty_cache()

In [3]:
"""
  Smart MCQ Solver — Bidirectional LSTM FROM SCRATCH

  Architecture:
    Token Embedding (random init, trained)
        ↓
    Bidirectional LSTM  (forward + backward pass over tokens)
        ↓
    Mean pooling of all hidden states
        ↓
    MLP head  (256 → 128 → 1 score per option)
        ↓
    Softmax over 5 option scores → Cross-Entropy loss
        ↓
    Adam optimiser (from scratch)
    5-Fold CV + W&B logging
"""

import os, math, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import wandb
warnings.filterwarnings("ignore")
os.environ["WANDB_SILENT"] = "true"

### CONFIG

In [4]:
DATA_DIR    = Path("/kaggle/input/competitions/smart-mcq-solver-challenge")
OPTION_COLS = ["A", "B", "C", "D", "E"]
N_FOLDS     = 5
SEED        = 42
np.random.seed(SEED)

VOCAB_SIZE   = 10_000   # top-N tokens
EMBED_DIM    = 64       # embedding dimension
HIDDEN_DIM   = 128      # LSTM hidden units (each direction)
MAX_SEQ_LEN  = 40       # truncate/pad to this length
MLP_DIMS     = [256, 128]
DROPOUT_RATE = 0.3
LR           = 5e-4
EPOCHS       = 20
BATCH_SIZE   = 32
LABEL_SMOOTH = 0.1
GRAD_CLIP    = 5.0      # gradient clipping for LSTM stability

### W&B

In [5]:
try:
    from kaggle_secrets import UserSecretsClient
    WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception:
    WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")

wandb.login(key=WANDB_API_KEY)
wandb.init(
    project="24f2000817-t22026",
    name="bilstm-scratch",
    config=dict(
        vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM, max_seq_len=MAX_SEQ_LEN,
        mlp_dims=MLP_DIMS, dropout=DROPOUT_RATE,
        lr=LR, epochs=EPOCHS, batch_size=BATCH_SIZE,
        label_smooth=LABEL_SMOOTH, grad_clip=GRAD_CLIP,
        n_folds=N_FOLDS,
    ),
)

### MAP@3

In [6]:
def apk(actual, predicted, k=3):
    if not actual: return 0.0
    score, hits = 0.0, 0
    for i, p in enumerate(predicted[:k]):
        if p in actual and p not in predicted[:i]:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(actual), k)

def mapk(actuals, predictions, k=3):
    return np.mean([apk([a], p, k) for a, p in zip(actuals, predictions)])

### DATA

In [7]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")
train_df = train_df.dropna(subset=["prompt"] + OPTION_COLS + ["answer"]).reset_index(drop=True)
test_df  = test_df.dropna(subset=["prompt"] + OPTION_COLS).reset_index(drop=True)
print(f"Train: {len(train_df)}  |  Test: {len(test_df)}")

label_map = {c: i for i, c in enumerate(OPTION_COLS)}
train_df["label_idx"] = train_df["answer"].map(label_map)

Train: 2000  |  Test: 500


### TOKENISER & VOCABULARY

In [8]:
PAD_IDX = 0
UNK_IDX = 1

def tokenise(text):
    return str(text).lower().split()

def build_vocab(texts, max_size):
    counter = Counter()
    for t in texts:
        counter.update(tokenise(t))
    vocab = {"<PAD>": PAD_IDX, "<UNK>": UNK_IDX}
    for word, _ in counter.most_common(max_size - 2):
        vocab[word] = len(vocab)
    return vocab

def encode(text, vocab, max_len):
    toks = tokenise(text)[:max_len]
    ids  = [vocab.get(t, UNK_IDX) for t in toks]
    # pad / truncate
    ids  = ids + [PAD_IDX] * (max_len - len(ids))
    return np.array(ids, dtype=np.int32)

# Build vocab from all texts
all_texts = []
for df_ in [train_df, test_df]:
    for _, row in df_.iterrows():
        all_texts.append(str(row["prompt"]))
        for c in OPTION_COLS:
            all_texts.append(str(row[c]))

vocab = build_vocab(all_texts, VOCAB_SIZE)
print(f"Vocabulary size: {len(vocab)}")

def encode_row(row):
    """Encode prompt + each option → (5, MAX_SEQ_LEN) token ids."""
    q    = str(row["prompt"])
    seqs = []
    for c in OPTION_COLS:
        # concatenate question and option as one sequence
        combined = q + " " + str(row[c])
        seqs.append(encode(combined, vocab, MAX_SEQ_LEN))
    return np.stack(seqs)   # (5, MAX_SEQ_LEN)

print("Encoding train ...")
X_train_tok = np.stack([encode_row(row) for _, row in train_df.iterrows()])  # (N, 5, L)
y_train     = train_df["label_idx"].values

print("Encoding test ...")
X_test_tok  = np.stack([encode_row(row) for _, row in test_df.iterrows()])   # (N, 5, L)

Vocabulary size: 3818
Encoding train ...
Encoding test ...


### LSTM CELL — GATES FROM SCRATCH

In [9]:
def sigmoid(x):  return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))
def tanh(x):     return np.tanh(np.clip(x, -30, 30))
def relu(x):     return np.maximum(0.0, x)
def relu_grad(x):return (x > 0).astype(np.float32)

def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def cross_entropy(probs, labels, smooth=0.1, n_cls=5):
    N = len(labels)
    sl = np.full((N, n_cls), smooth / n_cls, dtype=np.float32)
    sl[np.arange(N), labels] += 1 - smooth
    return -np.sum(sl * np.log(probs + 1e-9)) / N, sl


class LSTMCell:
    """
    Single LSTM cell for one time step.
    Weights: W_ih (input→hidden) and W_hh (hidden→hidden), bias b.
    Gate order: [i, f, g, o]  (input, forget, cell, output)
    """
    def __init__(self, input_dim, hidden_dim):
        H, D = hidden_dim, input_dim
        scale = np.sqrt(1.0 / H)
        # combined weight: [4H × (D+H)]
        self.W  = (np.random.randn(D + H, 4 * H) * scale).astype(np.float32)
        self.b  = np.zeros(4 * H, dtype=np.float32)
        self.H  = H
        # Adam moments
        self.mW = np.zeros_like(self.W)
        self.vW = np.zeros_like(self.W)
        self.mb = np.zeros_like(self.b)
        self.vb = np.zeros_like(self.b)
        self.cache = []   # per time step

    def step_forward(self, x, h_prev, c_prev):
        """x: (B, D), h_prev: (B, H), c_prev: (B, H)"""
        xh    = np.concatenate([x, h_prev], axis=1)  # (B, D+H)
        gates = xh @ self.W + self.b                  # (B, 4H)
        H     = self.H
        i_gate = sigmoid(gates[:, :H])
        f_gate = sigmoid(gates[:, H:2*H])
        g_gate = tanh(gates[:, 2*H:3*H])
        o_gate = sigmoid(gates[:, 3*H:])
        c_next = f_gate * c_prev + i_gate * g_gate
        tanh_c = tanh(c_next)
        h_next = o_gate * tanh_c
        self.cache.append((xh, i_gate, f_gate, g_gate, o_gate, c_prev, c_next, tanh_c))
        return h_next, c_next

    def seq_forward(self, X):
        """X: (B, T, D) → H_all: (B, T, H), last h and c"""
        B, T, _ = X.shape
        h = np.zeros((B, self.H), dtype=np.float32)
        c = np.zeros((B, self.H), dtype=np.float32)
        self.cache = []
        H_all = []
        for t in range(T):
            h, c = self.step_forward(X[:, t, :], h, c)
            H_all.append(h)
        return np.stack(H_all, axis=1), h, c   # (B,T,H), (B,H), (B,H)

    def seq_backward(self, dH_all, dh_last, dc_last):
        """
        dH_all : (B, T, H) — gradient from downstream for all time steps
        Returns dX: (B, T, D), dW, db
        """
        T  = dH_all.shape[1]
        H  = self.H
        dW = np.zeros_like(self.W)
        db = np.zeros_like(self.b)
        dh_next = dh_last.copy()
        dc_next = dc_last.copy()
        dX_all  = []

        for t in reversed(range(T)):
            xh, i_g, f_g, g_g, o_g, c_prev, c_next, tanh_c = self.cache[t]
            dh = dH_all[:, t, :] + dh_next

            # output gate
            do  = dh * tanh_c
            dc  = dh * o_g * (1 - tanh_c**2) + dc_next

            # cell gate
            df  = dc * c_prev
            di  = dc * g_g
            dg  = dc * i_g
            dc_prev = dc * f_g

            # gate pre-activations
            di_pre = di * i_g * (1 - i_g)
            df_pre = df * f_g * (1 - f_g)
            dg_pre = dg * (1 - g_g**2)
            do_pre = do * o_g * (1 - o_g)

            dgates = np.concatenate([di_pre, df_pre, dg_pre, do_pre], axis=1)  # (B,4H)
            dW    += xh.T @ dgates
            db    += dgates.sum(axis=0)
            dxh    = dgates @ self.W.T
            dX_all.append(dxh[:, :-H])   # strip dh_prev part
            dh_next = dxh[:, -H:]
            dc_next = dc_prev

        dX_all.reverse()
        return np.stack(dX_all, axis=1), dW, db

### BILSTM  (forward + backward LSTM)

In [10]:
class BiLSTM:
    def __init__(self, input_dim, hidden_dim):
        self.fwd = LSTMCell(input_dim, hidden_dim)
        self.bwd = LSTMCell(input_dim, hidden_dim)
        self.H   = hidden_dim

    def forward(self, X):
        """X: (B, T, D) → out: (B, T, 2H)"""
        H_fwd, _, _ = self.fwd.seq_forward(X)
        H_bwd, _, _ = self.bwd.seq_forward(X[:, ::-1, :])   # reverse seq
        H_bwd_rev   = H_bwd[:, ::-1, :]                      # flip back
        return np.concatenate([H_fwd, H_bwd_rev], axis=-1)   # (B, T, 2H)

    def backward(self, dout, mask=None):
        """dout: (B, T, 2H)"""
        H      = self.H
        d_fwd  = dout[:, :, :H]
        d_bwd  = dout[:, :, H:]
        dX_f, dW_f, db_f = self.fwd.seq_backward(d_fwd,
                             np.zeros((dout.shape[0], H), np.float32),
                             np.zeros((dout.shape[0], H), np.float32))
        dX_b, dW_b, db_b = self.bwd.seq_backward(d_bwd[:, ::-1, :],
                             np.zeros((dout.shape[0], H), np.float32),
                             np.zeros((dout.shape[0], H), np.float32))
        dX = dX_f + dX_b[:, ::-1, :]
        return dX, dW_f, db_f, dW_b, db_b

### EMBEDDING LAYER

In [11]:
class Embedding:
    def __init__(self, vocab_size, embed_dim):
        self.E    = (np.random.randn(vocab_size, embed_dim) * 0.01).astype(np.float32)
        self.mE   = np.zeros_like(self.E)
        self.vE   = np.zeros_like(self.E)
        self.idx  = None

    def forward(self, idx):
        """idx: (B, T) → (B, T, E)"""
        self.idx = idx
        return self.E[idx]

    def backward(self, dout):
        """dout: (B, T, E)"""
        dE = np.zeros_like(self.E)
        np.add.at(dE, self.idx, dout)
        return dE

### LINEAR + DROPOUT (for MLP head)

In [12]:
class Linear:
    def __init__(self, in_dim, out_dim):
        scale = np.sqrt(2.0 / in_dim)
        self.W  = (np.random.randn(in_dim, out_dim) * scale).astype(np.float32)
        self.b  = np.zeros(out_dim, dtype=np.float32)
        self.mW = np.zeros_like(self.W)
        self.vW = np.zeros_like(self.W)
        self.mb = np.zeros_like(self.b)
        self.vb = np.zeros_like(self.b)
        self.x  = None

    def forward(self, x):
        self.x = x
        return x @ self.W + self.b

    def backward(self, dout):
        dW = self.x.T @ dout
        db = dout.sum(axis=0)
        dx = dout @ self.W.T
        return dx, dW, db


class Dropout:
    def __init__(self, rate):
        self.rate = rate
        self.mask = None

    def forward(self, x, training=True):
        if training and self.rate > 0:
            self.mask = (np.random.rand(*x.shape) > self.rate).astype(np.float32)
            return x * self.mask / (1 - self.rate)
        return x

    def backward(self, dout):
        return dout * self.mask / (1 - self.rate)

### FULL MODEL

In [13]:
class BiLSTMModel:
    """
    token_ids (B, T) → Embedding (B,T,E) → BiLSTM (B,T,2H)
    → mean pool (B,2H) → Linear→ReLU→Dropout→Linear→ReLU→Linear(→1)
    """
    def __init__(self):
        self.embed   = Embedding(VOCAB_SIZE, EMBED_DIM)
        self.bilstm  = BiLSTM(EMBED_DIM, HIDDEN_DIM)
        # MLP head
        in_dim = HIDDEN_DIM * 2
        self.mlp     = []
        self.drops   = []
        prev = in_dim
        for h in MLP_DIMS:
            self.mlp.append(Linear(prev, h))
            self.drops.append(Dropout(DROPOUT_RATE))
            prev = h
        self.out = Linear(prev, 1)
        self.t   = 0   # Adam step
        # cache
        self.emb_out = self.lstm_out = self.pool = None
        self.pre_acts = []

    def forward(self, token_ids, training=True):
        """token_ids: (B, T) → score: (B, 1)"""
        x = self.embed.forward(token_ids)          # (B, T, E)
        h = self.bilstm.forward(x)                 # (B, T, 2H)
        # mean pooling (ignore PAD for cleanliness — simple mean here)
        p = h.mean(axis=1)                         # (B, 2H)
        self.emb_out, self.lstm_out, self.pool = x, h, p
        self.pre_acts = []
        for lin, drop in zip(self.mlp, self.drops):
            p = lin.forward(p)
            self.pre_acts.append(p.copy())
            p = relu(p)
            p = drop.forward(p, training)
        return self.out.forward(p)                 # (B, 1)

    def backward(self, dout):
        grads = {}
        dx, dW, db = self.out.backward(dout)
        grads["out_W"], grads["out_b"] = dW, db

        for i in reversed(range(len(self.mlp))):
            dx = self.drops[i].backward(dx)
            dx = dx * relu_grad(self.pre_acts[i])
            dx, dW, db = self.mlp[i].backward(dx)
            grads[f"mlp_W_{i}"] = dW
            grads[f"mlp_b_{i}"] = db

        # mean pool backward
        T  = self.lstm_out.shape[1]
        dh = np.repeat(dx[:, np.newaxis, :], T, axis=1) / T  # (B,T,2H)

        # BiLSTM backward
        dX, dW_f, db_f, dW_b, db_b = self.bilstm.backward(dh)
        grads["lstm_fwd_W"] = dW_f
        grads["lstm_fwd_b"] = db_f
        grads["lstm_bwd_W"] = dW_b
        grads["lstm_bwd_b"] = db_b

        # Embedding backward
        grads["embed"] = self.embed.backward(dX)
        return grads

    def clip_and_update(self, grads, lr, beta1=0.9, beta2=0.999, eps=1e-8):
        self.t += 1
        t = self.t

        # clip LSTM grads
        for key in ["lstm_fwd_W", "lstm_fwd_b", "lstm_bwd_W", "lstm_bwd_b"]:
            norm = np.linalg.norm(grads[key])
            if norm > GRAD_CLIP:
                grads[key] = grads[key] * GRAD_CLIP / norm

        def adam(param, grad, m, v):
            m[:] = beta1 * m + (1 - beta1) * grad
            v[:] = beta2 * v + (1 - beta2) * grad**2
            mh = m / (1 - beta1**t)
            vh = v / (1 - beta2**t)
            param -= lr * mh / (np.sqrt(vh) + eps)

        adam(self.out.W, grads["out_W"], self.out.mW, self.out.vW)
        adam(self.out.b, grads["out_b"], self.out.mb, self.out.vb)

        for i, lin in enumerate(self.mlp):
            adam(lin.W, grads[f"mlp_W_{i}"], lin.mW, lin.vW)
            adam(lin.b, grads[f"mlp_b_{i}"], lin.mb, lin.vb)

        adam(self.bilstm.fwd.W, grads["lstm_fwd_W"], self.bilstm.fwd.mW, self.bilstm.fwd.vW)
        adam(self.bilstm.fwd.b, grads["lstm_fwd_b"], self.bilstm.fwd.mb, self.bilstm.fwd.vb)
        adam(self.bilstm.bwd.W, grads["lstm_bwd_W"], self.bilstm.bwd.mW, self.bilstm.bwd.vW)
        adam(self.bilstm.bwd.b, grads["lstm_bwd_b"], self.bilstm.bwd.mb, self.bilstm.bwd.vb)

        # sparse embedding update
        dE   = grads["embed"]
        idx  = self.embed.idx
        uniq = np.unique(idx)
        for ui in uniq:
            mask = (idx == ui)
            g    = dE[ui]
            self.embed.mE[ui] = beta1 * self.embed.mE[ui] + (1 - beta1) * g
            self.embed.vE[ui] = beta2 * self.embed.vE[ui] + (1 - beta2) * g**2
            mh = self.embed.mE[ui] / (1 - beta1**t)
            vh = self.embed.vE[ui] / (1 - beta2**t)
            self.embed.E[ui] -= lr * mh / (np.sqrt(vh) + eps)

### SCORE ALL 5 OPTIONS

In [14]:
def score_options(model, X_tok_5, training=True):
    """X_tok_5: (N, 5, T) → logits: (N, 5)"""
    N = X_tok_5.shape[0]
    scores = np.zeros((N, 5), dtype=np.float32)
    for j in range(5):
        scores[:, j] = model.forward(X_tok_5[:, j, :], training).squeeze(-1)
    return scores

### TRAIN ONE FOLD

In [15]:
def train_fold(fold, tr_idx, vl_idx):
    print(f"\n{'='*50}\nFOLD {fold+1}/{N_FOLDS}\n{'='*50}")

    X_tr, y_tr = X_train_tok[tr_idx], y_train[tr_idx]
    X_vl, y_vl = X_train_tok[vl_idx], y_train[vl_idx]
    N_tr = len(X_tr)

    model       = BiLSTMModel()
    best_map3   = 0.0
    best_state  = None
    global_step = 0

    for epoch in range(EPOCHS):
        perm = np.random.permutation(N_tr)
        X_tr, y_tr = X_tr[perm], y_tr[perm]

        epoch_loss, n_correct, n_total = 0.0, 0, 0

        for start in range(0, N_tr, BATCH_SIZE):
            xb = X_tr[start:start + BATCH_SIZE]   # (B, 5, T)
            yb = y_tr[start:start + BATCH_SIZE]
            B  = len(xb)

            # forward: score each option
            scores = np.zeros((B, 5), dtype=np.float32)
            for j in range(5):
                scores[:, j] = model.forward(xb[:, j, :], training=True).squeeze(-1)

            probs = softmax(scores)
            loss, smooth_labels = cross_entropy(probs, yb, LABEL_SMOOTH)

            epoch_loss += loss * B
            n_correct  += (probs.argmax(axis=1) == yb).sum()
            n_total    += B

            # backward: accumulate grads across 5 options
            dscores = (probs - smooth_labels) / B   # (B, 5)
            total_grads = None

            for j in range(5):
                # re-run forward to rebuild cache for option j
                _ = model.forward(xb[:, j, :], training=True)
                dout_j  = dscores[:, j:j+1]
                grads_j = model.backward(dout_j)
                if total_grads is None:
                    total_grads = grads_j
                else:
                    for k in grads_j:
                        total_grads[k] = total_grads[k] + grads_j[k]

            model.clip_and_update(total_grads, LR)
            global_step += 1

            wandb.log({
                f"fold{fold+1}/train/step_loss": loss,
            }, step=global_step)

        epoch_loss /= n_total
        epoch_acc   = n_correct / n_total

        # validation
        val_scores = score_options(model, X_vl, training=False)
        val_preds  = [[OPTION_COLS[i] for i in np.argsort(r)[::-1]][:3]
                      for r in val_scores]
        val_map3   = mapk([OPTION_COLS[l] for l in y_vl], val_preds)

        print(f"  Ep{epoch+1:02d}  loss={epoch_loss:.4f}  "
              f"acc={epoch_acc:.4f}  val_MAP@3={val_map3:.4f}")

        wandb.log({
            f"fold{fold+1}/train/epoch_loss": epoch_loss,
            f"fold{fold+1}/train/epoch_acc":  epoch_acc,
            f"fold{fold+1}/val/map3":         val_map3,
            "epoch": epoch + 1,
        }, step=global_step)

        if val_map3 > best_map3:
            best_map3  = val_map3
            best_state = {
                "embed":   model.embed.E.copy(),
                "lstm_fW": model.bilstm.fwd.W.copy(),
                "lstm_fb": model.bilstm.fwd.b.copy(),
                "lstm_bW": model.bilstm.bwd.W.copy(),
                "lstm_bb": model.bilstm.bwd.b.copy(),
                "mlp_W":   [l.W.copy() for l in model.mlp],
                "mlp_b":   [l.b.copy() for l in model.mlp],
                "out_W":   model.out.W.copy(),
                "out_b":   model.out.b.copy(),
            }
            wandb.run.summary[f"fold{fold+1}/best_val_map3"] = best_map3
            print(f"         ✓ best saved (MAP@3={best_map3:.4f})")

    # restore best
    model.embed.E          = best_state["embed"]
    model.bilstm.fwd.W     = best_state["lstm_fW"]
    model.bilstm.fwd.b     = best_state["lstm_fb"]
    model.bilstm.bwd.W     = best_state["lstm_bW"]
    model.bilstm.bwd.b     = best_state["lstm_bb"]
    for i, lin in enumerate(model.mlp):
        lin.W = best_state["mlp_W"][i]
        lin.b = best_state["mlp_b"][i]
    model.out.W = best_state["out_W"]
    model.out.b = best_state["out_b"]

    test_scores = score_options(model, X_test_tok, training=False)
    return test_scores, best_map3

### 5-FOLD CV

In [16]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
all_test_scores, fold_scores = [], []

for fold, (tr_idx, vl_idx) in enumerate(skf.split(train_df, y_train)):
    scores, best = train_fold(fold, tr_idx, vl_idx)
    all_test_scores.append(scores)
    fold_scores.append(best)

print(f"\nCV MAP@3 per fold : {[f'{s:.4f}' for s in fold_scores]}")
print(f"Mean CV MAP@3     : {np.mean(fold_scores):.4f}")

wandb.run.summary["cv_map3_mean"] = float(np.mean(fold_scores))
wandb.run.summary["cv_map3_std"]  = float(np.std(fold_scores))
wandb.log({
    "cv_fold_map3": wandb.plot.bar(
        wandb.Table(
            columns=["fold", "val_map3"],
            data=[[f"fold{i+1}", s] for i, s in enumerate(fold_scores)],
        ),
        "fold", "val_map3", title="Val MAP@3 per Fold",
    )
})


FOLD 1/5
  Ep01  loss=1.6395  acc=0.3269  val_MAP@3=0.5754
         ✓ best saved (MAP@3=0.5754)
  Ep02  loss=2.5287  acc=0.3556  val_MAP@3=0.6679
         ✓ best saved (MAP@3=0.6679)
  Ep03  loss=2.3113  acc=0.5019  val_MAP@3=0.7450
         ✓ best saved (MAP@3=0.7450)
  Ep04  loss=1.4298  acc=0.6162  val_MAP@3=0.7842
         ✓ best saved (MAP@3=0.7842)
  Ep05  loss=1.3831  acc=0.6687  val_MAP@3=0.8087
         ✓ best saved (MAP@3=0.8087)
  Ep06  loss=1.1860  acc=0.7075  val_MAP@3=0.8187
         ✓ best saved (MAP@3=0.8187)
  Ep07  loss=1.3586  acc=0.7025  val_MAP@3=0.8371
         ✓ best saved (MAP@3=0.8371)
  Ep08  loss=1.1914  acc=0.7375  val_MAP@3=0.8363
  Ep09  loss=1.1346  acc=0.7519  val_MAP@3=0.8421
         ✓ best saved (MAP@3=0.8421)
  Ep10  loss=1.2887  acc=0.7406  val_MAP@3=0.8529
         ✓ best saved (MAP@3=0.8529)
  Ep11  loss=1.1629  acc=0.7631  val_MAP@3=0.8467
  Ep12  loss=1.0761  acc=0.7662  val_MAP@3=0.8529
         ✓ best saved (MAP@3=0.8529)
  Ep13  loss=1.1038 

### ENSEMBLE & SUBMISSION

In [17]:
avg_scores = np.mean(all_test_scores, axis=0)
test_preds = [[OPTION_COLS[i] for i in np.argsort(r)[::-1]][:3]
              for r in avg_scores]

submission = pd.DataFrame({
    "ID":         test_df["id"],
    "Prediction": [" ".join(p) for p in test_preds],
})
out_path = "/kaggle/working/submission_bilstm_scratch.csv"
submission.to_csv(out_path, index=False)
print(f"\n✅  Saved -> {out_path}")
print(submission.head(10).to_string(index=False))

wandb.finish()


✅  Saved -> /kaggle/working/submission_bilstm_scratch.csv
 ID Prediction
  1      A E B
  2      B A D
  3      B D C
  4      E C A
  5      C B D
  6      D C E
  7      E A D
  8      B C E
  9      C D A
 10      C B A
